# IMFACT vs Native Guide vs Wachter on FruitFlies

This notebook compares three counterfactual methods on **FruitFlies**:
- `imfact_cf` from `cfts.cf_imfact.imfact`
- `native_guide_uni_cf` from `cfts.cf_native_guide.native_guide`
- `wachter_gradient_cf` from `cfts.cf_wachter.wachter`

For IMFACT, it uses ablation-selected settings:
- `method="variance"`
- `n_nuns=3`
- `nun_switch="cycle"`

The notebook reports counterfactual quality metrics (`L2`, `DTW`, `sparsity`, `validity`) and visualizes UMAP projections where UMAP is fit on background dataset points and then used to transform query/counterfactual points.

In [1]:
from __future__ import annotations

from pathlib import Path
import random
import sys
import warnings
import urllib.request
import subprocess

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import umap
from aeon.datasets import load_classification
from scipy.spatial.distance import jensenshannon
from sklearn.metrics import f1_score
from sklearn.preprocessing import OneHotEncoder
from torch.utils.data import DataLoader

warnings.filterwarnings("ignore")

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "cfts").exists():
        sys.path.insert(0, str(candidate))
        sys.path.insert(0, str(candidate / "examples"))
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate repository root containing cfts/")

from base.data import TimeSeriesDataset, collate_sparse
from base.model import SimpleCNN
from cfts.cf_imfact.imfact import _sift_imfs, _psd, imfact_cf
from cfts.cf_wachter.wachter import wachter_gradient_cf
from cfts.metrics import dtw_distance, l2_distance, percentage_changed_points, prediction_change

plt.style.use("seaborn-v0_8-darkgrid")
np.random.seed(42)
random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Repo root: {REPO_ROOT}")

/workspaces/counterfactual-explanations-for-time-series/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu
Repo root: /workspaces/counterfactual-explanations-for-time-series


In [2]:
def to_channel_first(sample: np.ndarray) -> np.ndarray:
    arr = np.asarray(sample, dtype=np.float32)
    if arr.ndim == 1:
        return arr.reshape(1, -1)
    if arr.ndim == 2:
        return arr if arr.shape[0] <= arr.shape[1] else arr.T
    raise ValueError(f"Unsupported sample shape: {arr.shape}")


def to_class_index(label) -> int:
    label_arr = np.asarray(label)
    if label_arr.ndim == 0:
        return int(label_arr)
    return int(np.argmax(label_arr))


def predict_scores(model: torch.nn.Module, sample: np.ndarray, device: torch.device) -> np.ndarray:
    ts = torch.tensor(sample, dtype=torch.float32, device=device)
    if ts.ndim == 1:
        ts = ts.reshape(1, 1, -1)
    elif ts.ndim == 2:
        if ts.shape[0] > ts.shape[1]:
            ts = ts.T
        ts = ts.unsqueeze(0)
    with torch.no_grad():
        return model(ts).detach().cpu().numpy().reshape(-1)


def model_wrapper_factory(model: torch.nn.Module, device: torch.device):
    def wrapped(ts: np.ndarray) -> np.ndarray:
        return predict_scores(model, np.asarray(ts, dtype=np.float32), device)

    return wrapped


def select_correct_indices(model: torch.nn.Module, dataset, max_count: int, device: torch.device) -> list[int]:
    selected = []
    for idx in range(len(dataset)):
        sample, label = dataset[idx]
        scores = predict_scores(model, np.asarray(sample, dtype=np.float32), device)
        if int(np.argmax(scores)) == to_class_index(label):
            selected.append(idx)
        if len(selected) >= max_count:
            break
    return selected


def infer_target_class(scores: np.ndarray) -> int:
    order = np.argsort(scores)[::-1]
    return int(order[1])


def _select_native_guide_targeted(sample: np.ndarray, dataset, target_class: int) -> np.ndarray:
    sample_cf = to_channel_first(sample)
    sample_psd = _psd(sample_cf[0])

    candidates = []
    for idx in range(len(dataset)):
        cand_x, cand_y = dataset[idx]
        cand_label = to_class_index(cand_y)
        if cand_label != int(target_class):
            continue
        candidates.append(to_channel_first(cand_x))

    if not candidates:
        raise ValueError(f"No native guide found for target_class={target_class}")

    js_dists = [jensenshannon(sample_psd, _psd(candidate[0])) for candidate in candidates]
    best_idx = int(np.argmin(js_dists))
    return candidates[best_idx]


def run_imfact_variance(sample: np.ndarray, dataset, model: torch.nn.Module, target_class: int):
    return imfact_cf(
        sample=sample,
        dataset=dataset,
        model=model,
        method="variance",
        guide_class=target_class,
        step=0.05,
        max_iter=80,
        max_imfs=6,
        n_nuns=1,
        nun_switch="cycle",
        verbose=False,
    )


def run_native_guide(sample: np.ndarray, dataset, model: torch.nn.Module, target_class: int):
    sample_cf = to_channel_first(sample)
    device = next(model.parameters()).device

    def model_predict(arr):
        return predict_scores(model, np.asarray(arr, dtype=np.float32), device)

    sample_scores = model_predict(sample_cf.reshape(1, *sample_cf.shape))[0]
    sample_label = int(np.argmax(sample_scores))
    if sample_label == target_class:
        return sample_cf, sample_scores

    native_guide = _select_native_guide_targeted(sample_cf, dataset, target_class)
    guide_label = target_class
    candidate = sample_cf.copy()
    candidate_scores = sample_scores.copy()

    importance = np.abs(native_guide[0] - sample_cf[0])
    if np.allclose(importance.sum(), 0.0):
        importance = np.ones_like(importance)

    window = max(1, min(8, sample_cf.shape[1] // 10))
    for _ in range(128):
        if window >= sample_cf.shape[1]:
            start = 0
        else:
            conv = np.convolve(importance, np.ones(window, dtype=importance.dtype), mode="valid")
            start = int(np.argmax(conv))
        end = min(sample_cf.shape[1], start + window)
        candidate[:, start:end] = native_guide[:, start:end]
        candidate_scores = model_predict(candidate.reshape(1, *candidate.shape))[0]
        if int(np.argmax(candidate_scores)) == guide_label:
            break
        window = min(sample_cf.shape[1], window + 1)

    return candidate, candidate_scores


def run_wachter(sample: np.ndarray, dataset, model: torch.nn.Module, target_class: int):
    return wachter_gradient_cf(
        sample=sample,
        dataset=dataset,
        model=model,
        target=target_class,
        max_cfs=3,
        distance="euclidean",
        verbose=False,
    )

In [3]:
# FruitFlies config aligned with ablation setup.
DOWNSAMPLE = 1
N_BACKGROUND = 512


def _load_fruitflies_resilient(split: str):
    data_root = REPO_ROOT / "data" / "UCR"
    dataset_dir = data_root / "FruitFlies"
    train_ts = dataset_dir / "FruitFlies_TRAIN.ts"
    test_ts = dataset_dir / "FruitFlies_TEST.ts"

    if not (train_ts.exists() and test_ts.exists()):
        archive = data_root / "FruitFlies.zip"
        if not archive.exists():
            url = "https://timeseriesclassification.com/aeon-toolkit/FruitFlies.zip"
            print("Downloading FruitFlies archive...")
            urllib.request.urlretrieve(url, str(archive))

        print("Extracting FruitFlies .ts files...")
        dataset_dir.mkdir(parents=True, exist_ok=True)
        subprocess.run(
            [
                "unzip",
                "-j",
                "-o",
                str(archive),
                "FruitFlies_TRAIN.ts",
                "FruitFlies_TEST.ts",
                "-d",
                str(dataset_dir),
            ],
            check=True,
        )

    X, y = load_classification(name="FruitFlies", split=split, extract_path=str(data_root))
    encoder = OneHotEncoder(categories="auto", sparse_output=False)
    y_enc = encoder.fit_transform(np.expand_dims(y, axis=-1))
    dataset = TimeSeriesDataset(X=X, y=y_enc, name="FruitFlies", mapping=encoder.categories_)
    dataloader = DataLoader(dataset, batch_size=256, shuffle=False, collate_fn=collate_sparse)
    return dataloader, dataset


def _downsample_series(series: np.ndarray, factor: int) -> np.ndarray:
    arr = np.asarray(series, dtype=np.float32)
    if factor <= 1:
        return arr
    return arr[..., ::factor]


class _DownsampledView:
    def __init__(self, dataset, factor: int):
        self._dataset = dataset
        self._factor = factor

    def __len__(self):
        return len(self._dataset)

    def __getitem__(self, idx):
        x, y = self._dataset[idx]
        return _downsample_series(np.asarray(x, dtype=np.float32), self._factor), y

    def __getattr__(self, name):
        return getattr(self._dataset, name)


# Load FruitFlies and the matching pretrained model.
_, _dataset_train_raw = _load_fruitflies_resilient(split="train")
_, _dataset_test_raw = _load_fruitflies_resilient(split="test")

output_classes = _dataset_train_raw.y_shape[1]
raw_length = int(_dataset_train_raw.X_shape[2])
dataset_length = raw_length // DOWNSAMPLE if DOWNSAMPLE > 1 else raw_length

# Downsampled views are used by all downstream evaluation cells.
dataset_train = _DownsampledView(_dataset_train_raw, DOWNSAMPLE)
dataset_test = _DownsampledView(_dataset_test_raw, DOWNSAMPLE)

models_dir = REPO_ROOT / "models"
model_path = models_dir / f"simple_cnn_fruitflies_{output_classes}_len{dataset_length}.pth"

if not model_path.exists():
    raise FileNotFoundError(
        "Could not find FruitFlies checkpoint at "
        f"{model_path}. Train it with downsample={DOWNSAMPLE}."
    )

model = SimpleCNN(
    output_channels=output_classes,
    input_length=dataset_length,
).to(device)
state = torch.load(model_path, map_location=device)
model.load_state_dict(state)
print(f"Loaded model: {model_path}")

model.eval()

with torch.no_grad():
    y_true = []
    y_pred = []
    for sample, label in dataset_test:
        scores = predict_scores(model, np.asarray(sample, dtype=np.float32), device)
        y_true.append(to_class_index(label))
        y_pred.append(int(np.argmax(scores)))

test_f1 = f1_score(y_true, y_pred, average="macro")

print(f"Train size: {len(dataset_train)} | Test size: {len(dataset_test)}")
print(f"Model input length set to: {dataset_length}")
print(f"Test macro F1: {test_f1:.4f}")

Loaded model: /workspaces/counterfactual-explanations-for-time-series/models/simple_cnn_fruitflies_3_len5000.pth


Train size: 17259 | Test size: 17259
Model input length set to: 5000
Test macro F1: 0.8656


In [4]:
# Evaluate methods on correctly classified samples.
N_SAMPLES = 1
selected_indices = select_correct_indices(model, dataset_test, max_count=N_SAMPLES, device=device)
print(f"Using {len(selected_indices)} correctly classified samples.")

model_for_metrics = model_wrapper_factory(model, device)
records = []
all_cf_outputs = {}

for sample_rank, idx in enumerate(selected_indices):
    sample, label = dataset_test[idx]
    sample = np.asarray(sample, dtype=np.float32)

    scores_orig = predict_scores(model, sample, device)
    pred_orig = int(np.argmax(scores_orig))
    true_label = to_class_index(label)
    target_class = infer_target_class(scores_orig)

    runs = {
        "imfact_variance_nun3": lambda: run_imfact_variance(sample, dataset_test, model, target_class),
        "native_guide": lambda: run_native_guide(sample, dataset_test, model, target_class),
    }

    all_cf_outputs[idx] = {"sample": sample, "true_label": true_label, "target_class": target_class}

    for method_name, runner in runs.items():
        try:
            cf, pred_cf_scores = runner()
        except Exception as exc:
            records.append({
                "sample_idx": idx,
                "method": method_name,
                "true_label": true_label,
                "pred_orig": pred_orig,
                "target_class": target_class,
                "pred_cf": None,
                "success": False,
                "l2_norm": np.nan,
                "dtw_proximity": np.nan,
                "sparsity": np.nan,
                "validity": 0.0,
                "error": f"{type(exc).__name__}: {exc}",
            })
            continue

        if cf is None or pred_cf_scores is None:
            records.append({
                "sample_idx": idx,
                "method": method_name,
                "true_label": true_label,
                "pred_orig": pred_orig,
                "target_class": target_class,
                "pred_cf": None,
                "success": False,
                "l2_norm": np.nan,
                "dtw_proximity": np.nan,
                "sparsity": np.nan,
                "validity": 0.0,
                "error": "Method returned None",
            })
            continue

        cf = np.asarray(cf, dtype=np.float32)
        pred_cf = int(np.argmax(np.asarray(pred_cf_scores).reshape(-1)))
        success = pred_cf == target_class

        sample_cf = to_channel_first(sample)
        cf_cf = to_channel_first(cf)

        l2_val = float(l2_distance(sample_cf, cf_cf))
        dtw_val = float(dtw_distance(sample_cf, cf_cf))
        sparsity_val = float(1.0 - percentage_changed_points(sample_cf, cf_cf))
        validity_val = float(prediction_change(sample_cf, cf_cf, model_for_metrics, target_class=target_class))

        records.append({
            "sample_idx": idx,
            "method": method_name,
            "true_label": true_label,
            "pred_orig": pred_orig,
            "target_class": target_class,
            "pred_cf": pred_cf,
            "success": bool(success),
            "l2_norm": l2_val,
            "dtw_proximity": dtw_val,
            "sparsity": sparsity_val,
            "validity": validity_val,
            "error": None,
        })

        all_cf_outputs[idx][method_name] = cf

results_df = pd.DataFrame(records)
results_df.head()

Using 1 correctly classified samples.


In [ ]:
# UMAP projection for one representative evaluated sample.
# Use a capped, downsampled background for tractable fitting.
_rng = np.random.default_rng(42)
_bg_idx = sorted(_rng.choice(len(_dataset_test_raw), size=min(N_BACKGROUND, len(_dataset_test_raw)), replace=False).tolist())

background_data = np.stack([
    _downsample_series(to_channel_first(_dataset_test_raw[i][0]), DOWNSAMPLE).reshape(-1)
    for i in _bg_idx
], axis=0)
background_labels = np.array([to_class_index(_dataset_test_raw[i][1]) for i in _bg_idx])

method_order = ["imfact_variance_nun3", "native_guide", "wachter"]

sample_success_counts = (
    results_df[results_df["sample_idx"].isin(selected_indices)]
    .groupby("sample_idx")["success"]
    .sum()
    .sort_values(ascending=False)
 )

full_success_indices = [
    int(sample_idx) for sample_idx, count in sample_success_counts.items() if int(count) == len(method_order)
 ]

if full_success_indices:
    rep_idx = full_success_indices[0]
else:
    rep_idx = int(sample_success_counts.index[0])

rep_payload = all_cf_outputs[rep_idx]
rep_sample = to_channel_first(rep_payload["sample"]).reshape(1, -1)

n_neighbors = min(15, max(2, len(background_data) - 1))
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=n_neighbors,
    min_dist=0.15,
    metric="euclidean",
    random_state=42,
 )
background_emb = reducer.fit_transform(background_data)
sample_emb = reducer.transform(rep_sample)[0]

method_colors = {
    "imfact_variance_nun3": "#e63946",
    "native_guide": "#1d3557",
    "wachter": "#ff7f11",
}

sample_results = results_df[results_df["sample_idx"] == rep_idx]
metric_lookup = {
    row["method"]: {
        "l2": float(row["l2_norm"]),
        "dtw": float(row["dtw_proximity"]),
        "success": bool(row["success"]),
        "pred": int(row["pred_cf"]) if not pd.isna(row["pred_cf"]) else None,
    }
    for _, row in sample_results.iterrows()
}

orig_row = sample_results.iloc[0] if len(sample_results) else None
true_label = int(orig_row["true_label"]) if orig_row is not None else to_class_index(rep_payload["true_label"] if "true_label" in rep_payload else 0)
initial_pred = int(orig_row["pred_orig"]) if orig_row is not None else None

fig, ax = plt.subplots(figsize=(9, 7))

for class_id in np.unique(background_labels):
    mask = background_labels == class_id
    ax.scatter(
        background_emb[mask, 0],
        background_emb[mask, 1],
        s=10,
        alpha=0.25,
        label=f"background class {class_id}",
    )

ax.scatter(sample_emb[0], sample_emb[1], s=140, c="black", marker="o", edgecolors="white", label="original")

annotation_lines = [f"sample {rep_idx}", f"ground truth={true_label}", f"initial prediction={initial_pred}"]

for method_name in method_order:
    cf = rep_payload.get(method_name)
    if cf is None:
        continue
    cf_flat = to_channel_first(cf).reshape(1, -1)
    cf_emb = reducer.transform(cf_flat)[0]
    metrics = metric_lookup.get(method_name, {})
    l2_val = metrics.get("l2", np.nan)
    dtw_val = metrics.get("dtw", np.nan)
    worked = metrics.get("success", False)
    pred_cf = metrics.get("pred", None)

    annotation_lines.append(f"{method_name}: pred={pred_cf}, L2={l2_val:.3f}, DTW={dtw_val:.3f}, {'worked' if worked else 'failed'}")

    ax.scatter(
        cf_emb[0],
        cf_emb[1],
        s=150,
        c=method_colors[method_name],
        marker="X",
        edgecolors="white",
        linewidths=1.0,
        label=f"{method_name} (pred={pred_cf}, L2={l2_val:.2f}, DTW={dtw_val:.2f})",
    )
    ax.plot([sample_emb[0], cf_emb[0]], [sample_emb[1], cf_emb[1]], color=method_colors[method_name], alpha=0.5)

ax.text(
    0.02,
    0.98,
    "\n".join(annotation_lines),
    transform=ax.transAxes,
    va="top",
    ha="left",
    fontsize=8,
    bbox={"boxstyle": "round,pad=0.35", "facecolor": "white", "alpha": 0.8, "edgecolor": "none"},
)

ax.set_title(f"UMAP projection for sample {rep_idx}")
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
ax.grid(True, alpha=0.25)
ax.legend(loc="best", fontsize=8, ncols=2)
plt.tight_layout()
plt.show()

In [ ]:
# Signal-space comparison with subplots for the same representative sample.
method_order = ["imfact_variance_nun3", "native_guide", "wachter"]

sample_success_counts = (
    results_df[results_df["sample_idx"].isin(selected_indices)]
    .groupby("sample_idx")["success"]
    .sum()
    .sort_values(ascending=False)
 )

full_success_indices = [
    int(sample_idx) for sample_idx, count in sample_success_counts.items() if int(count) == len(method_order)
 ]

if full_success_indices:
    rep_idx = full_success_indices[0]
else:
    rep_idx = int(sample_success_counts.index[0])

rep_payload = all_cf_outputs[rep_idx]
x = to_channel_first(rep_payload["sample"])[0]
x_axis = np.arange(len(x))

n_rows = 1 + len(method_order)
fig, axes = plt.subplots(n_rows, 1, figsize=(14, 2.8 * n_rows), sharex=True)

# Build success and prediction lookup for this sample from evaluation records.
sample_results = results_df[results_df["sample_idx"] == rep_idx]
info_lookup = {
    row["method"]: {
        "success": bool(row["success"]),
        "pred": int(row["pred_cf"]) if not pd.isna(row["pred_cf"]) else None,
    }
    for _, row in sample_results.iterrows()
}

orig_row = sample_results.iloc[0] if len(sample_results) else None
true_label = int(orig_row["true_label"]) if orig_row is not None else None
initial_pred = int(orig_row["pred_orig"]) if orig_row is not None else None

status_parts = []
for method_name in method_order:
    worked = info_lookup.get(method_name, {}).get("success", False)
    status_parts.append(f"{method_name}: {'worked' if worked else 'failed'}")

fig.suptitle(
    f"Counterfactual waveforms for sample {rep_idx} | true={true_label} | initial pred={initial_pred} | " + " | ".join(status_parts),
    fontsize=12,
    y=0.995,
 )

# Top panel: original only.
axes[0].plot(x, label="original", color="black", linewidth=1.8, alpha=0.75)
axes[0].set_title(f"Original waveform for sample {rep_idx} (true={true_label}, initial pred={initial_pred})")
axes[0].set_ylabel("Amplitude")
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc="best")

# Remaining panels: one method per subplot.
for i, method_name in enumerate(method_order, start=1):
    ax = axes[i]
    cf = rep_payload.get(method_name)
    info = info_lookup.get(method_name, {})
    worked = info.get("success", False)
    pred_cf = info.get("pred", None)

    ax.plot(x, label="original", color="black", linewidth=1.0, linestyle="--", alpha=0.25)
    if cf is None:
        ax.text(0.5, 0.5, f"{method_name}: no counterfactual available", ha="center", va="center", transform=ax.transAxes)
    else:
        cf_line = to_channel_first(cf)[0]
        ax.plot(cf_line, label=method_name, linewidth=1.6, alpha=0.95, color="#1f77b4")
        ax.fill_between(x_axis, x, cf_line, color="#1f77b4", alpha=0.18, linewidth=0)
        diff_abs = np.abs(cf_line - x)
        top_idx = np.argsort(diff_abs)[-3:] if diff_abs.size >= 3 else np.arange(diff_abs.size)
        ax.scatter(x_axis[top_idx], cf_line[top_idx], s=18, color="#d62728", alpha=0.9, zorder=4)

    ax.set_title(f"{method_name} waveform (pred={pred_cf}, true={true_label}, {'worked' if worked else 'failed'})")
    ax.set_ylabel("Amplitude")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")

axes[-1].set_xlabel("Time step")
plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()